# Diagnóstico de SOP (Síndrome dos Ovários Policísticos) — PCOS
**Dataset:** PCOS Without Infertility ([Kaggle](https://www.kaggle.com/datasets/prasoonkottarathil/polycystic-ovary-syndrome-pcos))

**Objetivo:** Construir um modelo de **Regressão Logística** para prever a presença de SOP a partir de dados clínicos e laboratoriais, usando um subconjunto enxuto das **20 features mais correlacionadas com o diagnóstico**.

**Por que Regressão Logística com as Top 20 features?**
Após as etapas de comparação de modelos e de seleção de variáveis, esta combinação foi escolhida como solução final:

- **Regressão Logística** reúne as melhores características para um problema clínico: coeficientes interpretáveis (log-odds), boa generalização e compatibilidade com explicações exatas via SHAP linear.
- **Top 20 features** (selecionadas por correlação absoluta com o target) entregaram desempenho **igual ou superior** ao modelo com todas as features no conjunto de teste, com menos da metade das variáveis. Um modelo mais enxuto é mais fácil de interpretar, mais barato de operacionalizar na prática clínica (menos exames a coletar) e menos sujeito a sobreajuste por variáveis ruidosas.

> **Nota sobre este notebook:** versão *somente texto*. As saídas são tabelas e
> métricas numéricas — sem gráficos — priorizando leitura rápida, diffs limpos
> em controle de versão e execução leve. As mesmas informações antes mostradas
> em gráficos (distribuições, importância de features, desempenho) aparecem aqui
> em formato tabular.

**Estrutura do notebook:**
1. Importações
2. Funções auxiliares
3. Carregamento dos dados
4. Análise exploratória (EDA) e limpeza
5. Seleção das Top 20 features
6. Pré-processamento e pipeline (Top 20)
7. Divisão treino/teste
8. Avaliação por validação cruzada (5-fold)
9. Análise de overfitting (treino vs. validação)
10. Avaliação no conjunto de teste
11. Explicabilidade com SHAP


## 1. Importações

In [ ]:
# Importações centralizadas — apenas o necessário para Regressão Logística
import os

import numpy as np
import pandas as pd
import shap

import kagglehub

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Exibe todas as colunas do DataFrame ao imprimir
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Semente global para reprodutibilidade
RANDOM_STATE = 42

# Quantidade de features a selecionar por correlação com o target
N_TOP_FEATURES = 20


## 2. Funções auxiliares

Duas funções utilitárias reaproveitadas ao longo do notebook, ambas
específicas para a Regressão Logística:

- `avaliar_logreg` — imprime o relatório de classificação, o AUC-ROC e a matriz de confusão (em texto).
- `cross_validate_logreg` — roda validação cruzada 5-fold e reporta as métricas, incluindo o gap de overfitting (treino − validação).

In [ ]:
def avaliar_logreg(y_real, y_pred, y_proba, nome="Regressão Logística"):
    """
    Imprime métricas completas de classificação, o AUC-ROC e a matriz de
    confusão em formato textual.

    Parâmetros
    ----------
    y_real  : array-like — rótulos verdadeiros
    y_pred  : array-like — predições do modelo
    y_proba : array-like — probabilidades da classe positiva
    nome    : str        — identificador usado no cabeçalho
    """
    print(f"\n{'─' * 40}")
    print(f"  {nome}")
    print(f"{'─' * 40}")
    print(classification_report(y_real, y_pred))
    print(f"AUC-ROC: {roc_auc_score(y_real, y_proba):.4f}\n")

    # Matriz de confusão em texto
    cm = confusion_matrix(y_real, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=['Real: Não', 'Real: Sim'],
        columns=['Prev: Não', 'Prev: Sim'],
    )
    print("Matriz de confusão:")
    print(cm_df.to_string())


def cross_validate_logreg(modelo, pipeline_base, X_tr, y_tr, cv=5):
    """
    Executa validação cruzada k-fold para a Regressão Logística e
    reporta as métricas médias, incluindo o gap de overfitting.

    Retorna um dicionário com as métricas agregadas.
    """
    scoring = {
        "roc_auc":   "roc_auc",
        "f1":        "f1",
        "precision": "precision",
        "recall":    "recall",
        "accuracy":  "accuracy",
    }

    pipe = Pipeline([
        ('preprocessor', pipeline_base),
        ('model', modelo),
    ])

    cv_res = cross_validate(
        pipe, X_tr, y_tr,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
    )

    train_auc = cv_res['train_roc_auc'].mean()
    val_auc   = cv_res['test_roc_auc'].mean()
    gap       = train_auc - val_auc
    flag      = '🔴' if gap > 0.10 else ('🟡' if gap > 0.05 else '🟢')

    resultado = {
        'roc_auc':   val_auc,
        'f1':        cv_res['test_f1'].mean(),
        'precision': cv_res['test_precision'].mean(),
        'recall':    cv_res['test_recall'].mean(),
        'accuracy':  cv_res['test_accuracy'].mean(),
        'train_auc': train_auc,
        'gap':       gap,
        'flag':      flag,
    }

    print(f"\n📊 Regressão Logística — Cross-Validation ({cv}-fold)")
    print(f"  ROC AUC (val):   {val_auc:.4f}")
    print(f"  ROC AUC (train): {train_auc:.4f}")
    print(f"  Gap overfitting: {gap:.4f}  {flag}")
    print(f"  Recall:          {resultado['recall']:.4f}")
    print(f"  F1-score:        {resultado['f1']:.4f}")
    print(f"  Precision:       {resultado['precision']:.4f}")
    print(f"  Accuracy:        {resultado['accuracy']:.4f}")

    return resultado


## 3. Carregamento dos dados

In [ ]:
# Download automático do dataset via KaggleHub
path = kagglehub.dataset_download(
    "prasoonkottarathil/polycystic-ovary-syndrome-pcos"
)

df = pd.read_excel(
    os.path.join(path, "PCOS_data_without_infertility.xlsx"),
    sheet_name="Full_new"
)

print(f"Shape: {df.shape}")
df.head()


Using Colab cache for faster access to the 'polycystic-ovary-syndrome-pcos' dataset.
Shape: (541, 45)


,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),Hb(g/dl),Cycle(R/I),Cycle length(days),Marraige Status (Yrs),Pregnant(Y/N),No. of aborptions,I beta-HCG(mIU/mL),II beta-HCG(mIU/mL),FSH(mIU/mL),LH(mIU/mL),FSH/LH,Hip(inch),Waist(inch),Waist:Hip Ratio,TSH (mIU/L),AMH(ng/mL),PRL(ng/mL),Vit D3 (ng/mL),PRG(ng/mL),RBS(mg/dl),Weight gain(Y/N),hair growth(Y/N),Skin darkening (Y/N),Hair loss(Y/N),Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm),Unnamed: 44
0,1,1,0,28,44.6,152.0,19.300000,15,78,22,10.48,2,5,7.0,0,0,1.99,1.99,7.95,3.68,2.160326,36,30,0.833333,0.68,2.07,45.16,17.1,0.57,92.0,0,0,0,0,0,1.0,0,110,80,3,3,18.0,18.0,8.5,NaN
1,2,2,0,36,65.0,161.5,24.921163,15,74,20,11.70,2,5,11.0,1,0,60.80,1.99,6.73,1.09,6.174312,38,32,0.842105,3.16,1.53,20.09,61.3,0.97,92.0,0,0,0,0,0,0.0,0,120,70,3,5,15.0,14.0,3.7,NaN
2,3,3,1,33,68.8,165.0,25.270891,11,72,18,11.80,2,5,10.0,1,0,494.08,494.08,5.54,0.88,6.295455,40,36,0.900000,2.54,6.63,10.52,49.7,0.36,84.0,0,0,0,1,1,1.0,0,120,80,13,15,18.0,20.0,10.0,NaN
3,4,4,0,37,65.0,148.0,29.674945,13,72,20,12.00,2,5,4.0,0,0,1.99,1.99,8.06,2.36,3.415254,42,36,0.857143,16.41,1.22,36.90,33.4,0.36,76.0,0,0,0,0,0,0.0,0,120,70,2,2,15.0,14.0,7.5,NaN
4,5,5,0,25,52.0,161.0,20.060954,11,72,18,10.00,2,5,1.0,1,0,801.45,801.45,3.98,0.90,4.422222,37,30,0.810811,3.57,2.26,30.09,43.8,0.38,84.0,0,0,0,1,0,0.0,0,120,80,3,4,16.0,14.0,7.0,NaN


## 4. Análise exploratória (EDA) e limpeza

### 4.1 Visão geral do dataset

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 45 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Sl. No                  541 non-null    int64  
 1   Patient File No.        541 non-null    int64  
 2   PCOS (Y/N)              541 non-null    int64  
 3    Age (yrs)              541 non-null    int64  
 4   Weight (Kg)             541 non-null    float64
 5   Height(Cm)              541 non-null    float64
 6   BMI                     541 non-null    float64
 7   Blood Group             541 non-null    int64  
 8   Pulse rate(bpm)         541 non-null    int64  
 9   RR (breaths/min)        541 non-null    int64  
 10  Hb(g/dl)                541 non-null    float64
 11  Cycle(R/I)              541 non-null    int64  
 12  Cycle length(days)      541 non-null    int64  
 13  Marraige Status (Yrs)   540 non-null    float64
 14  Pregnant(Y/N)           541 non-null    in

### 4.2 Valores nulos
Contagem de valores faltantes por coluna (apenas colunas afetadas), usada para
guiar o tratamento na etapa seguinte.

In [ ]:
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

if len(nulos):
    print("Valores nulos por coluna (somente colunas afetadas):")
    print(nulos.to_string())
    print(f"\nTotal de células nulas: {int(nulos.sum())}")
else:
    print("Nenhum valor nulo encontrado.")


Valores nulos por coluna (somente colunas afetadas):
Unnamed: 44              539
Marraige Status (Yrs)      1
Fast food (Y/N)            1

Total de células nulas: 541


### 4.3 Tratamento de colunas problemáticas

**`Unnamed: 44`** — coluna sem padrão identificável, provavelmente um artefato da exportação do Excel. Removida.

**`Marraige Status (Yrs)`** — variável numérica; valores faltantes preenchidos com a **média** (distribuição contínua).

**`Fast food (Y/N)`** — variável binária; valores faltantes preenchidos com a **moda** para preservar a distribuição original.

**`II beta-HCG(mIU/mL)` e `AMH(ng/mL)`** — colunas lidas como `object` por conterem um valor corrompido. Convertidas com `pd.to_numeric` e nulos preenchidos com a **mediana** (robusta a outliers).

**`Sl. No` e `Patient File No.`** — identificadores sem valor preditivo. Removidos.

In [ ]:
# Remove coluna suja sem padrão identificável
df.drop(columns=['Unnamed: 44'], inplace=True)

# Marraige Status — preenche nulos com a média (valor contínuo)
# Obs.: reatribuição direta (em vez de inplace encadeado) evita
# ChainedAssignmentError nas versões recentes do pandas.
df['Marraige Status (Yrs)'] = df['Marraige Status (Yrs)'].fillna(
    df['Marraige Status (Yrs)'].mean()
)

# Fast food — preenche nulos com a moda (variável binária)
moda_fastfood = int(df['Fast food (Y/N)'].mode()[0])
df['Fast food (Y/N)'] = df['Fast food (Y/N)'].fillna(moda_fastfood)

# Converte colunas numéricas corrompidas e preenche com a mediana
for col in ['II    beta-HCG(mIU/mL)', 'AMH(ng/mL)']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# Remove identificadores
df.drop(columns=['Sl. No', 'Patient File No.'], inplace=True)

# Confirma que não restam nulos
print(f"Nulos restantes: {df.isnull().sum().sum()}")
df.info()


Nulos restantes: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 42 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   PCOS (Y/N)              541 non-null    int64  
 1    Age (yrs)              541 non-null    int64  
 2   Weight (Kg)             541 non-null    float64
 3   Height(Cm)              541 non-null    float64
 4   BMI                     541 non-null    float64
 5   Blood Group             541 non-null    int64  
 6   Pulse rate(bpm)         541 non-null    int64  
 7   RR (breaths/min)        541 non-null    int64  
 8   Hb(g/dl)                541 non-null    float64
 9   Cycle(R/I)              541 non-null    int64  
 10  Cycle length(days)      541 non-null    int64  
 11  Marraige Status (Yrs)   541 non-null    float64
 12  Pregnant(Y/N)           541 non-null    int64  
 13  No. of aborptions       541 non-null    int64  
 14    I   beta-HCG(mIU/mL) 

### 4.4 Codificação de variáveis categóricas
O campo `Blood Group` é uma variável categórica nominal (tipo sanguíneo),
não ordinal. Aplicamos **dummy encoding** (`get_dummies`) com `drop_first=True`
para evitar multicolinearidade perfeita.

In [ ]:
df = pd.get_dummies(df, columns=['Blood Group'], drop_first=True, dtype=int)
print(f"Shape após dummies: {df.shape}")
df.head()


Shape após dummies: (541, 48)


,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Pulse rate(bpm),RR (breaths/min),Hb(g/dl),Cycle(R/I),Cycle length(days),Marraige Status (Yrs),Pregnant(Y/N),No. of aborptions,I beta-HCG(mIU/mL),II beta-HCG(mIU/mL),FSH(mIU/mL),LH(mIU/mL),FSH/LH,Hip(inch),Waist(inch),Waist:Hip Ratio,TSH (mIU/L),AMH(ng/mL),PRL(ng/mL),Vit D3 (ng/mL),PRG(ng/mL),RBS(mg/dl),Weight gain(Y/N),hair growth(Y/N),Skin darkening (Y/N),Hair loss(Y/N),Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm),Blood Group_12,Blood Group_13,Blood Group_14,Blood Group_15,Blood Group_16,Blood Group_17,Blood Group_18
0,0,28,44.6,152.0,19.300000,78,22,10.48,2,5,7.0,0,0,1.99,1.99,7.95,3.68,2.160326,36,30,0.833333,0.68,2.07,45.16,17.1,0.57,92.0,0,0,0,0,0,1.0,0,110,80,3,3,18.0,18.0,8.5,0,0,0,1,0,0,0
1,0,36,65.0,161.5,24.921163,74,20,11.70,2,5,11.0,1,0,60.80,1.99,6.73,1.09,6.174312,38,32,0.842105,3.16,1.53,20.09,61.3,0.97,92.0,0,0,0,0,0,0.0,0,120,70,3,5,15.0,14.0,3.7,0,0,0,1,0,0,0
2,1,33,68.8,165.0,25.270891,72,18,11.80,2,5,10.0,1,0,494.08,494.08,5.54,0.88,6.295455,40,36,0.900000,2.54,6.63,10.52,49.7,0.36,84.0,0,0,0,1,1,1.0,0,120,80,13,15,18.0,20.0,10.0,0,0,0,0,0,0,0
3,0,37,65.0,148.0,29.674945,72,20,12.00,2,5,4.0,0,0,1.99,1.99,8.06,2.36,3.415254,42,36,0.857143,16.41,1.22,36.90,33.4,0.36,76.0,0,0,0,0,0,0.0,0,120,70,2,2,15.0,14.0,7.5,0,1,0,0,0,0,0
4,0,25,52.0,161.0,20.060954,72,18,10.00,2,5,1.0,1,0,801.45,801.45,3.98,0.90,4.422222,37,30,0.810811,3.57,2.26,30.09,43.8,0.38,84.0,0,0,0,1,0,0.0,0,120,80,3,4,16.0,14.0,7.0,0,0,0,0,0,0,0


### 4.5 Distribuição do target (PCOS)

In [ ]:
def modelo(a, b, metadados = False) :
  metadados_resultado = []
  if( metadados) # Se for true
    metadados.append("proporcao_classes", json.dumps(metadados) )
    metadados_resultado.append(metadados)
  return metadados_resultado


In [ ]:
contagem = df['PCOS (Y/N)'].value_counts().rename({0: 'Não', 1: 'Sim'})
proporcao = df['PCOS (Y/N)'].value_counts(normalize=True).rename({0: 'Não', 1: 'Sim'})

dist_target = pd.DataFrame({
    'Quantidade': contagem,
    'Proporção (%)': (proporcao * 100).round(1),
})
print("Distribuição do target — PCOS (Y/N):")
print(dist_target.to_string())


Distribuição do target — PCOS (Y/N):
            Quantidade  Proporção (%)
PCOS (Y/N)                           
Não                364           67.3
Sim                177           32.7


### 4.6 Pares de features altamente correlacionadas (multicolinearidade)
Features com correlação > 0.8 entre si podem ser redundantes e introduzir
instabilidade nos coeficientes de modelos lineares como a Regressão Logística.
Identificamos e removemos as mais óbvias **antes** de selecionar as Top 20, para
que a seleção não escolha duas variáveis que carregam essencialmente a mesma informação.

In [ ]:
# Identificar pares com |correlação| > 0.8 (excluindo autocorrelação)
corr_matrix = df.corr(numeric_only=True).abs()

# Máscara para ignorar a diagonal (autocorrelação = 1) sem mutar o array.
# Obs.: df.corr().values é somente-leitura em versões recentes do numpy/pandas,
# então usamos uma máscara em vez de np.fill_diagonal direto sobre os valores.
mask = np.ones(corr_matrix.shape, dtype=bool)
np.fill_diagonal(mask, False)

corr_pairs = (corr_matrix.where(mask)
              .unstack()
              .dropna()
              .drop_duplicates()
              .sort_values(ascending=False))

THRESHOLD = 0.8
high_corr = corr_pairs[corr_pairs > THRESHOLD]

print(f"Pares com |correlação| > {THRESHOLD}:")
for (c1, c2), val in high_corr.items():
    print(f"  {c1}  ×  {c2}:  {val:.2f}")


Pares com |correlação| > 0.8:
  FSH(mIU/mL)  ×  FSH/LH:  0.97
  Weight (Kg)  ×  BMI:  0.90
  Hip(inch)  ×  Waist(inch):  0.87


In [ ]:
# Remoção das features redundantes identificadas acima:
#   - 'Weight (Kg)' e 'Waist(inch)': substituídas pelo IMC (BMI) já presente
#   - 'FSH(mIU/mL)': altamente correlacionada com FSH/LH (calculada a partir dela)
df.drop(columns=['Weight (Kg)', 'FSH(mIU/mL)', 'Waist(inch)'], inplace=True)

print(f"Shape após remoção de redundâncias: {df.shape}")


Shape após remoção de redundâncias: (541, 45)


## 5. Seleção das Top 20 features

Selecionamos as **20 variáveis com maior correlação absoluta** com o target.
Esta é a base do modelo final: todas as etapas seguintes (pré-processamento,
treino, avaliação e explicabilidade) usam exclusivamente este subconjunto.

### 5.1 Ranking de correlação e seleção

In [ ]:
# Correlação de Pearson de cada feature com o target
corr_target = df.corr(numeric_only=True)['PCOS (Y/N)'].sort_values(ascending=False)
corr_abs    = corr_target.abs().drop('PCOS (Y/N)').sort_values(ascending=False)

# Top N features por correlação absoluta — usadas em TODO o restante do notebook
top_features = corr_abs.head(N_TOP_FEATURES).index.tolist()

# Tabela com o ranking, valor absoluto e sinal da correlação
ranking_top = pd.DataFrame({
    'feature':   top_features,
    '|corr|':    [round(corr_abs[f], 4) for f in top_features],
    'corr':      [round(corr_target[f], 4) for f in top_features],
    'sentido':   ['positiva' if corr_target[f] >= 0 else 'negativa' for f in top_features],
})
ranking_top.index = range(1, len(ranking_top) + 1)

print(f'Top {N_TOP_FEATURES} features selecionadas por |correlação| com PCOS:')
print(ranking_top.to_string())


Top 20 features selecionadas por |correlação| com PCOS:
                  feature  |corr|    corr   sentido
1        Follicle No. (R)  0.6483  0.6483  positiva
2        Follicle No. (L)  0.6033  0.6033  positiva
3    Skin darkening (Y/N)  0.4757  0.4757  positiva
4        hair growth(Y/N)  0.4647  0.4647  positiva
5        Weight gain(Y/N)  0.4410  0.4410  positiva
6              Cycle(R/I)  0.4016  0.4016  positiva
7         Fast food (Y/N)  0.3762  0.3762  positiva
8            Pimples(Y/N)  0.2861  0.2861  positiva
9              AMH(ng/mL)  0.2641  0.2641  positiva
10                    BMI  0.1995  0.1995  positiva
11     Cycle length(days)  0.1785 -0.1785  negativa
12         Hair loss(Y/N)  0.1729  0.1729  positiva
13              Age (yrs)  0.1685 -0.1685  negativa
14              Hip(inch)  0.1623  0.1623  positiva
15   Avg. F size (L) (mm)  0.1330  0.1330  positiva
16  Marraige Status (Yrs)  0.1127 -0.1127  negativa
17       Endometrium (mm)  0.1066  0.1066  positiva
18   Avg

### 5.2 Estatísticas descritivas das Top 20 por classe
Resumo numérico de como cada feature selecionada se comporta nas duas classes
(PCOS = Não vs. Sim). Substitui os gráficos de distribuição (violin/histograma):
a diferença entre as médias por classe indica o poder discriminativo da variável.

In [ ]:
# Média de cada feature do Top 20 por classe + diferença absoluta
desc_por_classe = (
    df.groupby('PCOS (Y/N)')[top_features]
    .mean()
    .T
    .rename(columns={0: 'Média (Não)', 1: 'Média (Sim)'})
)
desc_por_classe['Diferença'] = (
    desc_por_classe['Média (Sim)'] - desc_por_classe['Média (Não)']
)
desc_por_classe = desc_por_classe.round(3)

print("Média por classe — Top 20 features:")
print(desc_por_classe.to_string())


Média por classe — Top 20 features:
PCOS (Y/N)             Média (Não)  Média (Sim)  Diferença
Follicle No. (R)             4.637       10.763      6.125
Follicle No. (L)             4.352        9.785      5.434
Skin darkening (Y/N)         0.154        0.621      0.468
hair growth(Y/N)             0.129        0.571      0.442
Weight gain(Y/N)             0.228        0.684      0.456
Cycle(R/I)                   2.308        3.079      0.771
Fast food (Y/N)              0.385        0.785      0.401
Pimples(Y/N)                 0.390        0.695      0.305
AMH(ng/mL)                   4.539        7.845      3.305
BMI                         23.747       25.471      1.724
Cycle length(days)           5.126        4.559     -0.567
Hair loss(Y/N)               0.393        0.576      0.183
 Age (yrs)                  32.066       30.124     -1.942
Hip(inch)                   37.544       38.915      1.371
Avg. F size (L) (mm)        14.688       15.698      1.010
Marraige Status (Yrs

### 5.3 Correlação entre as Top 20 (multicolinearidade interna)

In [ ]:
# Maiores correlações entre pares de features DENTRO do Top 20
# (ajuda a entender redundância remanescente no subconjunto final)
corr_top = df[top_features].corr().abs()

# Máscara para ignorar a diagonal sem mutar o array (somente-leitura)
mask_top = np.ones(corr_top.shape, dtype=bool)
np.fill_diagonal(mask_top, False)

pares_top = (corr_top.where(mask_top)
             .unstack()
             .dropna()
             .drop_duplicates()
             .sort_values(ascending=False)
             .head(10))

print("Top 10 pares de features mais correlacionadas dentro do Top 20:")
for (c1, c2), val in pares_top.items():
    print(f"  {c1}  ×  {c2}:  {val:.2f}")


Top 10 pares de features mais correlacionadas dentro do Top 20:
  Follicle No. (R)  ×  Follicle No. (L):  0.80
   Age (yrs)  ×  Marraige Status (Yrs):  0.66
  BMI  ×  Hip(inch):  0.60
  Avg. F size (L) (mm)  ×  Avg. F size (R) (mm):  0.52
  Weight gain(Y/N)  ×  BMI:  0.46
  Skin darkening (Y/N)  ×  hair growth(Y/N):  0.37
  Weight gain(Y/N)  ×  Fast food (Y/N):  0.37
  Skin darkening (Y/N)  ×  Weight gain(Y/N):  0.35
  Skin darkening (Y/N)  ×  Fast food (Y/N):  0.34
  Follicle No. (R)  ×  Skin darkening (Y/N):  0.32


## 6. Pré-processamento e pipeline (Top 20)

Dentro das 20 features selecionadas, separamos dois grupos para tratamentos distintos:

- **Numéricas contínuas** → imputação pela mediana + normalização (`StandardScaler`)
- **Binárias (0/1)** → passadas sem transformação (`passthrough`)

A normalização é particularmente importante para a Regressão Logística: coloca
todas as variáveis na mesma escala, o que estabiliza a otimização e torna os
coeficientes comparáveis entre si.

Usamos `ColumnTransformer` + `Pipeline` do scikit-learn para garantir que o
`fit` aconteça **somente nos dados de treino**, eliminando data leakage.

In [ ]:
# Lista de referência de TODAS as colunas binárias do dataset
# (flags clínicas + dummies de tipo sanguíneo). Serve para classificar
# quais das Top 20 são binárias e quais são numéricas contínuas.
binary_cols_ref = [
    'Pregnant(Y/N)', 'Weight gain(Y/N)', 'hair growth(Y/N)',
    'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)',
    'Fast food (Y/N)', 'Reg.Exercise(Y/N)',
    'Blood Group_12', 'Blood Group_13', 'Blood Group_14',
    'Blood Group_15', 'Blood Group_16', 'Blood Group_17', 'Blood Group_18',
]

# Particiona as Top 20 em binárias e numéricas
binary_cols = [c for c in top_features if c in binary_cols_ref]
num_cols    = [c for c in top_features if c not in binary_cols_ref]

# Pipeline para numéricas: mediana → normalização z-score
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# Pipeline completo: transforma numéricas e passa binárias sem alteração
full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('bin', 'passthrough', binary_cols),
])

print(f"Numéricas no Top {N_TOP_FEATURES} ({len(num_cols)}): {num_cols}")
print(f"\nBinárias no Top {N_TOP_FEATURES}  ({len(binary_cols)}): {binary_cols}")


Numéricas no Top 20 (14): ['Follicle No. (R)', 'Follicle No. (L)', 'Cycle(R/I)', 'AMH(ng/mL)', 'BMI', 'Cycle length(days)', ' Age (yrs)', 'Hip(inch)', 'Avg. F size (L) (mm)', 'Marraige Status (Yrs)', 'Endometrium (mm)', 'Avg. F size (R) (mm)', 'Pulse rate(bpm) ', 'Hb(g/dl)']

Binárias no Top 20  (6): ['Skin darkening (Y/N)', 'hair growth(Y/N)', 'Weight gain(Y/N)', 'Fast food (Y/N)', 'Pimples(Y/N)', 'Hair loss(Y/N)']


## 7. Divisão treino/teste

Usamos `stratify=y` para garantir que a proporção de classes seja preservada
nos dois conjuntos (especialmente importante com dados levemente desbalanceados).

`X` contém **apenas as Top 20 features** selecionadas na Seção 5.

In [ ]:
X = df[top_features]
y = df['PCOS (Y/N)'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Aplica o pipeline SEM VAZAMENTO: fit apenas no treino, transform em ambos
X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared  = full_pipeline.transform(X_test)

print(f"Treino: {X_train_prepared.shape}  |  Teste: {X_test_prepared.shape}")

# Verificar balanceamento de classes nos dois conjuntos
train_pct = (y_train.value_counts(normalize=True) * 100).round(1)
test_pct  = (y_test.value_counts(normalize=True) * 100).round(1)
df_split = pd.DataFrame(
    {'Treino (%)': train_pct, 'Teste (%)': test_pct}
).sort_index()
df_split.index = df_split.index.map({0: 'Não', 1: 'Sim'})

print("\nDistribuição de PCOS — Treino vs Teste:")
print(df_split.to_string())


Treino: (432, 20)  |  Teste: (109, 20)

Distribuição de PCOS — Treino vs Teste:
            Treino (%)  Teste (%)
PCOS (Y/N)                       
Não               67.4       67.0
Sim               32.6       33.0


## 8. Avaliação por validação cruzada (5-fold)

Avaliamos a Regressão Logística com validação cruzada estratificada de 5 folds
sobre o conjunto de treino. Isso fornece uma estimativa estável do desempenho
antes de tocar no conjunto de teste.

**Configuração do modelo:**
- `max_iter=1000` — garante a convergência do otimizador.
- `class_weight='balanced'` — compensa o desbalanceamento penalizando mais os
  erros na classe minoritária (importante em diagnóstico, onde o recall da
  classe positiva é crítico).

In [ ]:
# Modelo final do notebook — Regressão Logística sobre as Top 20 features
logreg = LogisticRegression(max_iter=1000, class_weight='balanced')

print("=" * 60)
print(f"CROSS-VALIDATION (5-fold) — TOP {N_TOP_FEATURES} FEATURES")
print("=" * 60)

resultados_cv = cross_validate_logreg(logreg, full_pipeline, X_train, y_train)


CROSS-VALIDATION (5-fold) — TOP 20 FEATURES

📊 Regressão Logística — Cross-Validation (5-fold)
  ROC AUC (val):   0.9461
  ROC AUC (train): 0.9698
  Gap overfitting: 0.0237  🟢
  Recall:          0.8788
  F1-score:        0.8171
  Precision:       0.7649
  Accuracy:        0.8727


## 9. Análise de overfitting (treino vs. validação)

O gap entre o desempenho no treino e na validação é um indicador de overfitting:

- **Gap > 10%** 🔴 → overfitting forte — considerar mais regularização
- **Gap 5–10%** 🟡 → overfitting moderado — monitorar
- **Gap < 5%**  🟢 → generalização adequada

In [ ]:
train_auc = resultados_cv['train_auc']
val_auc   = resultados_cv['roc_auc']
gap       = resultados_cv['gap']

print(f"ROC AUC treino:     {train_auc:.4f}")
print(f"ROC AUC validação:  {val_auc:.4f}")
print(f"Gap de overfitting: {gap:.4f}  {resultados_cv['flag']}")


ROC AUC treino:     0.9698
ROC AUC validação:  0.9461
Gap de overfitting: 0.0237  🟢


## 10. Avaliação no conjunto de teste

Treinamos a Regressão Logística no conjunto de treino completo (Top 20 features)
e avaliamos no conjunto de teste — dados que o modelo nunca viu durante a
validação cruzada. O AUC-ROC resume a capacidade de separação entre as classes.

In [ ]:
# Pipeline final treinado em todo o conjunto de treino
pipe_final = Pipeline([
    ('preprocessor', full_pipeline),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
pipe_final.fit(X_train, y_train)

y_proba = pipe_final.predict_proba(X_test)[:, 1]
y_pred  = pipe_final.predict(X_test)

resultados_teste = {
    'roc_auc':   roc_auc_score(y_test, y_proba),
    'f1':        f1_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall':    recall_score(y_test, y_pred),
    'accuracy':  accuracy_score(y_test, y_pred),
}

# Relatório completo + matriz de confusão (texto)
avaliar_logreg(y_test, y_pred, y_proba,
               nome=f"Regressão Logística — Test Set (Top {N_TOP_FEATURES})")

print(f"\nMétricas no Test Set (Top {N_TOP_FEATURES}):")
metricas_tst = pd.Series(resultados_teste).round(4)
print(metricas_tst.to_string())



────────────────────────────────────────
  Regressão Logística — Test Set (Top 20)
────────────────────────────────────────
              precision    recall  f1-score   support

           0       0.96      0.90      0.93        73
           1       0.82      0.92      0.87        36

    accuracy                           0.91       109
   macro avg       0.89      0.91      0.90       109
weighted avg       0.91      0.91      0.91       109

AUC-ROC: 0.9513

Matriz de confusão:
           Prev: Não  Prev: Sim
Real: Não         66          7
Real: Sim          3         33

Métricas no Test Set (Top 20):
roc_auc      0.9513
f1           0.8684
precision    0.8250
recall       0.9167
accuracy     0.9083


## 11. Explicabilidade — SHAP

SHAP (SHapley Additive exPlanations) quantifica a contribuição individual de
cada feature para cada predição do modelo.

Usamos `LinearExplainer`, otimizado para modelos lineares como a Regressão
Logística — ele calcula contribuições lineares exatas usando a média como
baseline. Reportamos a **importância global** de cada feature como a média do
valor absoluto dos SHAP values (`|SHAP| médio`): quanto maior, mais a variável
pesa nas predições.

In [ ]:
# Modelo final já treinado (Regressão Logística, Top 20) — reutiliza pipe_final
# Transformar dados de treino para alimentar o explainer
X_train_transformed = pipe_final.named_steps['preprocessor'].transform(X_train)

# Nomes das features após o ColumnTransformer
feature_names = pipe_final.named_steps['preprocessor'].get_feature_names_out()

# LinearExplainer é o mais adequado para Regressão Logística:
# usa a média como baseline e calcula contribuições lineares exatas
explainer   = shap.LinearExplainer(
    pipe_final.named_steps['model'],
    X_train_transformed,
    feature_names=feature_names,
)
shap_values = explainer(X_train_transformed)

# Ranking numérico de importância global (|SHAP| médio)
shap_importance = (
    pd.DataFrame({
        'feature':    feature_names,
        'importance': np.abs(shap_values.values).mean(axis=0),
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
shap_importance.index = range(1, len(shap_importance) + 1)
shap_importance['importance'] = shap_importance['importance'].round(4)

print(f"Importância global das features (|SHAP| médio) — Top {N_TOP_FEATURES}:")
print(shap_importance.to_string())


Importância global das features (|SHAP| médio) — Top 20:
                       feature  importance
1        num__Follicle No. (R)      1.3123
2              num__Cycle(R/I)      0.6091
3        bin__hair growth(Y/N)      0.5850
4    bin__Skin darkening (Y/N)      0.5542
5        num__Follicle No. (L)      0.4782
6        bin__Weight gain(Y/N)      0.4345
7            bin__Pimples(Y/N)      0.4211
8   num__Marraige Status (Yrs)      0.2999
9        num__Endometrium (mm)      0.2022
10             num__AMH(ng/mL)      0.1766
11       num__Pulse rate(bpm)       0.1575
12   num__Avg. F size (L) (mm)      0.1538
13        bin__Fast food (Y/N)      0.1350
14               num__Hb(g/dl)      0.1151
15     num__Cycle length(days)      0.1064
16                    num__BMI      0.0944
17   num__Avg. F size (R) (mm)      0.0917
18         bin__Hair loss(Y/N)      0.0839
19              num__Hip(inch)      0.0827
20             num__ Age (yrs)      0.0068


### 11.1 Conclusão

A análise SHAP deixa claro que o modelo está apostando em sinais clássicos de
PCOS para tomar suas decisões — o que é ótimo para explicar os resultados. O
ranking de importância mostra que o **número de folículos**, o **ciclo
irregular**, o **crescimento de pelos** e o **escurecimento da pele** estão
entre as variáveis que mais pesam na previsão. O sinal dos coeficientes (e a
direção das correlações reportadas na Seção 5) reforça que, quanto mais altos
esses valores, maior a chance de o modelo apontar PCOS.

Em resumo, a **Regressão Logística treinada sobre as 20 features mais
correlacionadas** entrega desempenho forte e estável, foca em fatores
reconhecidos pela medicina e usa menos da metade das variáveis originais —
tornando as decisões fáceis de entender por médicos e pacientes e o modelo
mais simples de operacionalizar. Exatamente o tipo de transparência e economia
que justifica essa escolha para um problema de diagnóstico clínico.